In [1]:
# =========================================================
# HYBRID SARIMA + XGBOOST RESIDUAL MODEL
# =========================================================

# IDEA:
# 1. SARIMA captures:
#    - trend
#    - seasonality
#    - linear temporal structure
#
# 2. XGBoost models SARIMA residuals:
#    - nonlinear effects
#    - local irregularities
#    - extreme fluctuations
#
# FINAL PREDICTION:
#    Hybrid = SARIMA + XGB(residuals)
#
# Usually stronger than:
#    RF
#    XGB alone
#    SARIMA alone
# for hydrological monthly forecasting.

# =========================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from statsmodels.tsa.statespace.sarimax import SARIMAX
from xgboost import XGBRegressor

from sklearn.metrics import mean_squared_error, mean_absolute_error

# =========================================================
# METRICS
# =========================================================

def nse(y_true, y_pred):

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    denom = np.sum((y_true - np.mean(y_true))**2)

    if denom < 1e-12:
        return np.nan

    return 1 - np.sum((y_true - y_pred)**2) / denom


def compute_metrics(y_true, y_pred):

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    nse_val = nse(y_true, y_pred)

    return rmse, mae, nse_val

# =========================================================
# PREPARE TIME SERIES
# =========================================================

def prepare_time_series(df, value_name='flow'):

    df = df.copy()

    df.columns = df.columns.str.strip().str.lower()

    months = [
        "jan","feb","mar","apr","maj","jun",
        "jul","avg","sep","okt","nov","dec"
    ]

    existing_months = [m for m in months if m in df.columns]

    # numeric
    df[existing_months] = (
        df[existing_months]
        .apply(pd.to_numeric, errors='coerce')
    )

    # fill missing
    df[existing_months] = (
        df[existing_months]
        .ffill()
        .bfill()
    )

    # long format
    ts = df.melt(
        id_vars='year',
        value_vars=existing_months,
        var_name='month',
        value_name=value_name
    )

    ts['date'] = pd.to_datetime(
        ts['year'].astype(str) + '-' + ts['month'],
        format='%Y-%b',
        errors='coerce'
    )

    ts = ts.dropna(subset=['date'])

    ts = ts.set_index('date').sort_index()

    ts = ts.dropna(subset=[value_name])

    ts[value_name] = ts[value_name].clip(lower=0)

    return ts

# =========================================================
# CREATE SLIDING WINDOWS FOR RESIDUAL MODEL
# =========================================================

def create_windows(series, window_size=12):

    X = []
    y = []

    for i in range(window_size, len(series)):

        X.append(series[i-window_size:i])
        y.append(series[i])

    return np.array(X), np.array(y)

# =========================================================
# HYBRID SARIMA + XGB
# =========================================================

def hybrid_sarima_xgb(
    ts,
    label,
    window_size=12,
    split_date='2014-01-01',

    sarima_order=(1,1,1),
    sarima_seasonal=(1,1,1,12),

    xgb_config=None
):

    if xgb_config is None:

        xgb_config = {
            'n_estimators': 500,
            'max_depth': 4,
            'learning_rate': 0.03,
            'subsample': 0.8,
            'colsample_bytree': 0.8
        }

    # -----------------------------------------------------
    # 1. SERIES
    # -----------------------------------------------------

    series = ts['flow'].values.astype(float)

    series = np.maximum(series, 0)

    dates = ts.index

    split_date = pd.to_datetime(split_date)

    split_idx = np.where(dates < split_date)[0][-1]

    train_series = series[:split_idx]
    test_series = series[split_idx:]

    test_dates = dates[split_idx:]

    # -----------------------------------------------------
    # 2. SARIMA
    # -----------------------------------------------------

    print("\nTraining SARIMA...")

    sarima_model = SARIMAX(
        train_series,

        order=sarima_order,
        seasonal_order=sarima_seasonal,

        enforce_stationarity=False,
        enforce_invertibility=False
    )

    sarima_fit = sarima_model.fit(disp=False)

    # -----------------------------------------------------
    # 3. SARIMA RESIDUALS
    # -----------------------------------------------------

    residuals = sarima_fit.resid

    residuals = np.nan_to_num(residuals)

    # -----------------------------------------------------
    # 4. XGB TRAINING DATA
    # -----------------------------------------------------

    X_res, y_res = create_windows(
        residuals,
        window_size=window_size
    )

    # -----------------------------------------------------
    # 5. TRAIN XGB ON RESIDUALS
    # -----------------------------------------------------

    print("Training XGBoost on SARIMA residuals...")

    xgb_model = XGBRegressor(

        n_estimators=xgb_config['n_estimators'],
        max_depth=xgb_config['max_depth'],
        learning_rate=xgb_config['learning_rate'],
        subsample=xgb_config['subsample'],
        colsample_bytree=xgb_config['colsample_bytree'],

        objective='reg:squarederror',

        random_state=42,
        n_jobs=-1
    )

    xgb_model.fit(X_res, y_res)

    # -----------------------------------------------------
    # 6. FORECAST
    # -----------------------------------------------------

    print("Forecasting...")

    sarima_forecast = sarima_fit.forecast(
        steps=len(test_series)
    )

    # iterative residual prediction
    residual_history = list(residuals[-window_size:])

    residual_preds = []

    for i in range(len(test_series)):

        x_input = np.array(
            residual_history[-window_size:]
        ).reshape(1, -1)

        pred_residual = xgb_model.predict(x_input)[0]

        residual_preds.append(pred_residual)

        residual_history.append(pred_residual)

    residual_preds = np.array(residual_preds)

    # -----------------------------------------------------
    # 7. HYBRID PREDICTION
    # -----------------------------------------------------

    hybrid_pred = sarima_forecast + residual_preds

    hybrid_pred = np.maximum(hybrid_pred, 0)

    # -----------------------------------------------------
    # 8. METRICS
    # -----------------------------------------------------

    rmse, mae, nse_val = compute_metrics(
        test_series,
        hybrid_pred
    )

    print("\n==============================")
    print(f"{label} - Hybrid SARIMA + XGB")
    print("==============================")
    print(f"RMSE: {rmse:.2f}")
    print(f"MAE : {mae:.2f}")
    print(f"NSE : {nse_val:.4f}")

    # -----------------------------------------------------
    # 9. PLOTS
    # -----------------------------------------------------

    plt.figure(figsize=(14,5))

    plt.plot(
        test_dates,
        test_series,
        label='Actual'
    )

    plt.plot(
        test_dates,
        hybrid_pred,
        label='Hybrid SARIMA+XGB',
        marker='x'
    )

    plt.title(f"{label} Hybrid Forecast (2014–2025)")
    plt.xlabel("Date")
    plt.ylabel("Flow")

    plt.legend()
    plt.grid(True)

    plt.show()

    # -----------------------------------------------------
    # 10. RETURN
    # -----------------------------------------------------

    return {
        "predictions": hybrid_pred,
        "actual": test_series,
        "dates": test_dates,
        "sarima_model": sarima_fit,
        "xgb_model": xgb_model,
        "rmse": rmse,
        "mae": mae,
        "nse": nse_val
    }

In [ ]:
# =========================================================
# LOAD DATA
# =========================================================

q_min = pd.read_csv("q_min.csv")

ts_qmin = prepare_time_series(q_min)

# =========================================================
# CONFIG
# =========================================================

xgb_config = {
    'n_estimators': 1000,
    'max_depth': 4,
    'learning_rate': 0.03,
    'subsample': 0.8,
    'colsample_bytree': 0.8
}

# =========================================================
# RUN HYBRID MODEL
# =========================================================

results = hybrid_sarima_xgb(
    ts_qmin,
    label="Q_MIN",

    window_size=12,

    sarima_order=(1, 1, 1),
    sarima_seasonal=(1,3,3,12),

    xgb_config=xgb_config
)


Training SARIMA...
